# TIGER Pipeline — Full End-to-End Demo

This notebook shows how a non-specialist user can:

1. set up paths
2. run **Step 1** mutational search on a **custom template**
3. run **Step 2** toxicity filtering
4. (optionally) prepare inputs for **Step 3** HelixFold
5. report the **full classification metric suite** (Acc / P / R / F1 / MCC / AUC-ROC / AUC-PR) on demo labeled sets

> Use the `tiger-pipeline` / `ccseg` kernel for Steps 1–2.  
> HelixFold / Rosetta cells are gated behind flags and should be run in their own environments.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PIPELINE = Path('..').resolve() if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
# If launched from notebooks/, parent is pipeline root; if launched from pipeline/, use cwd.
if (Path.cwd() / '01_mutation_search').exists():
    PIPELINE = Path.cwd()
elif (Path.cwd().parent / '01_mutation_search').exists():
    PIPELINE = Path.cwd().parent
else:
    PIPELINE = Path('/data4T/ubuntu/wangyue/postdoc_2025/TIGER_3D/TIGER/pipeline')

sys.path.insert(0, str(PIPELINE))
print('PIPELINE =', PIPELINE)
assert (PIPELINE / '01_mutation_search' / 'search_mutations.py').exists()


## A. Custom template configuration

Change `TEMPLATE` to any amino-acid sequence you want to explore.


In [ ]:
TEMPLATE = 'KSMLKSMK'   # <-- replace with your peptide
K = 1                   # mutation depth (1 is fastest for demos)
RUN_HELIXFOLD = False   # set True only inside HelixFold env
RUN_RELAX = False       # set True only inside PyRosetta env

step1_out = PIPELINE / '01_mutation_search' / 'outputs' / 'full_demo'
step2_out = PIPELINE / '02_toxicity_filter' / 'outputs' / 'full_demo'
step3_pdb = PIPELINE / '03_structure_prediction' / 'outputs' / 'pdb' / 'full_demo'
step3_relax = PIPELINE / '03_structure_prediction' / 'outputs' / 'relaxed' / 'full_demo'
print(TEMPLATE, K)


## B. Step 1 — Mutational search + activity filter


In [ ]:
import os
os.chdir(PIPELINE / '01_mutation_search')
from search_mutations import run_search

pos_csv, neg_csv = run_search(TEMPLATE, K, step1_out)
print('positives ->', pos_csv)
print('negatives ->', neg_csv)
pos = pd.read_csv(pos_csv, header=None, names=['seq','copy']) if pos_csv.stat().st_size else pd.DataFrame(columns=['seq','copy'])
print('n_positives =', len(pos))
display(pos.head())


## C. Step 2 — Toxicity filter

If Step 1 produced zero positives (possible for tiny demos), we fall back to the bundled sample input so the rest of the notebook still runs.


In [ ]:
os.chdir(PIPELINE / '02_toxicity_filter')
from types import SimpleNamespace
from infer_toxin import run_inference

input_csv = pos_csv if pos_csv.exists() and pos_csv.stat().st_size > 0 else (PIPELINE / '02_toxicity_filter' / 'examples' / 'sample_input.csv')
print('Using', input_csv)

args = SimpleNamespace(
    csv=str(input_csv),
    out_dir=str(step2_out),
    model_path=str(PIPELINE / '02_toxicity_filter' / 'checkpoints' / 'model_1.pth'),
    threshold=0.5,
    max_len=30,
    batch_size=32,
    q_encoder='gru',
    v_encoder='resnet34',
    channels=32,
    mode='101',
    cpu=True,
)
toxin_path, non_toxin_path, all_path = run_inference(args)
display(pd.read_csv(all_path).sort_values('tox_prob', ascending=False).head())
print('non-toxins ->', non_toxin_path)


## D. Step 3 — HelixFold / Relax (optional)

These cells are disabled by default. Activate the HelixFold kernel/env, set `RUN_HELIXFOLD=True`, and re-run.


In [ ]:
if RUN_HELIXFOLD:
    os.chdir(PIPELINE / '03_structure_prediction')
    from infer_batch import run_batch
    from types import SimpleNamespace
    hf_args = SimpleNamespace(
        csv_file=str(non_toxin_path),
        output_dir=str(step3_pdb),
        init_model=str(PIPELINE / '03_structure_prediction' / 'weights' / 'helixfold-single.pdparams'),
        skip_exist=True,
    )
    run_batch(hf_args)
else:
    print('Skipped HelixFold. Set RUN_HELIXFOLD=True in a HelixFold environment to enable.')

if RUN_RELAX:
    os.chdir(PIPELINE / '03_structure_prediction')
    from relax import run_relax
    run_relax(step3_pdb, step3_relax, workers=2)
else:
    print('Skipped relax. Set RUN_RELAX=True in a PyRosetta environment to enable.')


## E. Classification metrics on labeled demo sets

Reports **Accuracy, Precision, Recall, F1, MCC, AUC-ROC, AUC-PR** for all models across CV folds and the held-out test set.


In [ ]:
os.chdir(PIPELINE / '01_mutation_search')
!python evaluate_models.py   --csv examples/labeled_activity_demo.csv   --out-dir outputs/metrics_full_demo   --n-splits 5 --test-size 0.2

act_summary = pd.read_csv('outputs/metrics_full_demo/summary_metrics.csv')
print('Activity classifiers')
display(act_summary)


In [ ]:
os.chdir(PIPELINE / '02_toxicity_filter')
!python evaluate_models.py   --csv examples/labeled_toxicity_demo.csv   --out-dir outputs/metrics_full_demo   --n-splits 5 --test-size 0.2 --cpu

tox_summary = pd.read_csv('outputs/metrics_full_demo/summary_metrics.csv')
print('Toxicity classifiers')
display(tox_summary)


## F. Applying this to your own data

1. Put sequences/labels in CSV files with clear column names (`sequence`, `label`).
2. For discovery: pass `--sequence YOURSEQ` to Step 1, then chain CSVs through Steps 2–3.
3. For evaluation: point `evaluate_models.py --csv` at your labeled file.

See `docs/CUSTOM_DATA.md` and `docs/SETUP.md` for complete details.
